# 03 - Customer / Territory Segmentation

Build customer-level features and apply K-Means clustering to identify segments:
key accounts (bulk/high-value), high-potential retail, underserved territories, etc.

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
from eda_utils import load_clean_data, save_fig
from segmentation import (
    add_order_type, filter_valid_sales, build_customer_features,
    scale_features, find_optimal_k, run_kmeans, profile_clusters,
    plot_cluster_geo, plot_cluster_scatter
)

In [ ]:
df = load_clean_data("../data/processed/cleaned_data.csv")
df = filter_valid_sales(df)
df = add_order_type(df, bulk_quantile=0.99)
df.head()

## 1. Build customer-level features

In [ ]:
cust_df = build_customer_features(df)
print(f"Number of unique customers: {len(cust_df)}")
cust_df.head()

In [ ]:
cust_df.describe()

## 2. Select features and scale

Using: total_sales, total_quantity, n_transactions, avg_transaction_value, n_product_classes, bulk_order_share, sales_growth.

Note: total_sales is highly skewed (a few large key accounts) - consider log-transforming before scaling.

In [ ]:
import numpy as np

feature_cols = [
    'total_sales', 'total_quantity', 'n_transactions',
    'avg_transaction_value', 'n_product_classes', 'bulk_order_share', 'sales_growth'
]

# Log-transform skewed monetary/volume features (add 1 to avoid log(0))
cluster_df = cust_df.copy()
for col in ['total_sales', 'total_quantity', 'avg_transaction_value']:
    cluster_df[col] = np.log1p(cluster_df[col])

X, scaler = scale_features(cluster_df, feature_cols)
X.shape

## 3. Find optimal number of clusters

In [ ]:
fig, inertias, silhouettes = find_optimal_k(X, k_range=range(2, 8))
save_fig(fig, '10_cluster_optimal_k', path='../reports/figures')
plt.show()
print('Inertias:', inertias)
print('Silhouette scores:', silhouettes)

## 4. Fit K-Means (choose k based on elbow/silhouette above)

In [ ]:
K = 4  # adjust based on step 3 results
cust_df, km_model = run_kmeans(X, cust_df, k=K)
cust_df['cluster'].value_counts()

## 5. Profile each cluster

In [ ]:
profile = profile_clusters(cust_df, feature_cols)
profile

## 6. Visualize clusters

In [ ]:
fig = plot_cluster_geo(cust_df)
save_fig(fig, '11_clusters_geo', path='../reports/figures')
plt.show()

In [ ]:
fig = plot_cluster_scatter(cust_df, x='total_sales', y='bulk_order_share')
save_fig(fig, '12_clusters_sales_vs_bulkshare', path='../reports/figures')
plt.show()

In [ ]:
fig = plot_cluster_scatter(cust_df, x='total_sales', y='sales_growth')
save_fig(fig, '13_clusters_sales_vs_growth', path='../reports/figures')
plt.show()

## 7. Channel mix per cluster

In [ ]:
import seaborn as sns

channel_mix = pd.crosstab(cust_df['cluster'], cust_df['primary_channel'], normalize='index') * 100
channel_mix

In [ ]:
country_mix = pd.crosstab(cust_df['cluster'], cust_df['country'], normalize='index') * 100
country_mix

## 8. Name and interpret clusters

Based on the profile table above, assign business-friendly names, e.g.:
- **Key Accounts**: high total_sales, high bulk_order_share
- **Growth Retail**: moderate sales, high sales_growth, low bulk share
- **Stable Core**: moderate sales, low growth, broad product mix
- **Underserved / Long Tail**: low sales, low transactions, low product diversity

Map cluster numbers to names below.

In [ ]:
# Update this mapping after reviewing the profile table
cluster_names = {
    0: 'Segment A',
    1: 'Segment B',
    2: 'Segment C',
    3: 'Segment D',
}
cust_df['segment_name'] = cust_df['cluster'].map(cluster_names)
cust_df['segment_name'].value_counts()

## 9. Save segmented customer data

In [ ]:
cust_df.to_csv('../data/processed/customer_segments.csv', index=False)
print('Saved customer_segments.csv')

## Key Observations (fill in after running)

- Optimal k chosen: ...
- Cluster sizes and % of revenue: ...
- Cluster characteristics (key accounts vs retail vs underserved): ...
- Geographic patterns by cluster: ...
- Business recommendations per segment: ...